# 14_transiciones_segun_ivlr
## Análisis descriptivo de la relación entre IVLR inicial y destinos laborales en t+1

**Reglas de oro:**
- Usar exclusivamente el entorno Conda `trabajo`.

**Objetivo:** Analizar descriptivamente la asociación entre el IVLR observado en el trimestre de origen (t) y el estado laboral en t+1, restringiendo a empleo informal en t y ventana 2023T4–2025T3.

**Salidas:** Tablas en `Salidas/10_ivlr_transiciones/`.

In [1]:
# 1. Configuración y rutas
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(
    os.getenv(
        "TFM_PROJECT_ROOT",
        r"C:\Users\tomas\OneDrive\Master Granada\TFM",
    )
).expanduser().resolve()

PANEL_PATH = PROJECT_ROOT / "Salidas" / "06_transiciones" / "panel_estados_auditado.rds"
IVLR_PATH = PROJECT_ROOT / "Salidas" / "09_ivlr" / "base_ivlr_preliminar.parquet"
OUT_DIR = PROJECT_ROOT / "Salidas" / "10_ivlr_transiciones"
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert PANEL_PATH.exists(), f"Panel no encontrado: {PANEL_PATH}"
assert IVLR_PATH.exists(), f"IVLR no encontrado: {IVLR_PATH}"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"OUT_DIR: {OUT_DIR}")

PROJECT_ROOT: C:\Users\tomas\OneDrive\Master Granada\TFM
OUT_DIR: C:\Users\tomas\OneDrive\Master Granada\TFM\Salidas\10_ivlr_transiciones


In [2]:
# 2. Lectura robusta del panel (.rds)
def leer_rds(ruta: Path) -> pd.DataFrame:
    """Lee un .rds de R. Usa pyreadr si está instalado; si no, Rscript + arrow."""
    ruta = ruta.resolve()
    if not ruta.exists():
        raise FileNotFoundError(f"No existe: {ruta}")

    try:
        import pyreadr
        resultado = pyreadr.read_r(str(ruta))
        df = list(resultado.values())[0]
        if isinstance(df, pd.DataFrame):
            return df
    except Exception as e:
        print(f"Aviso: pyreadr no disponible: {e}")

    # Fallback: Rscript + arrow (requiere R y arrow instalados)
    import subprocess
    import tempfile
    with tempfile.TemporaryDirectory() as td:
        tmp = Path(td) / "panel.parquet"
        codigo = (
            "suppressMessages(library(arrow)); "
            f"x <- readRDS({str(ruta)!r}); "
            f"write_parquet(as.data.frame(x), {str(tmp)!r})"
        )
        proc = subprocess.run(["Rscript", "-e", codigo], capture_output=True, text=True)
        if proc.returncode != 0 or not tmp.exists():
            raise RuntimeError("No se pudo leer el .rds con Rscript.\n" + proc.stderr)
        return pd.read_parquet(tmp)

panel = leer_rds(PANEL_PATH)
print(f"Panel cargado: {panel.shape[0]} filas, {panel.shape[1]} columnas")

Panel cargado: 283246 filas, 38 columnas


In [3]:
# 3. Cargar IVLR preliminar
ivlr = pd.read_parquet(IVLR_PATH)
print(f"IVLR cargado: {ivlr.shape[0]} filas, {ivlr.shape[1]} columnas")

IVLR cargado: 80300 filas, 12 columnas


In [4]:
# 4. Preparación del panel: consistencia, id_persona, tipos
# Usar solo enlaces consistentes (como en scripts previos)
if "consistencia" in panel.columns:
    panel = panel[panel["consistencia"] == True].copy()
    print(f"Panel restringido a consistentes: {panel.shape[0]} filas")

# Construir id_persona
panel["id_persona"] = (
    panel["CODUSU"].astype(str) + "_" +
    panel["NRO_HOGAR"].astype(str) + "_" +
    panel["COMPONENTE"].astype(str)
)

# Convertir variables temporales a numéricas (Int64 nullable)
for col in ["ANO4", "TRIMESTRE", "ANO4_t1", "TRIMESTRE_t1"]:
    if col in panel.columns:
        panel[col] = pd.to_numeric(panel[col], errors="coerce").astype("Int64")

Panel restringido a consistentes: 283246 filas


In [5]:
# 5. Definir estados y ventana
estado_t = "estado_laboral_clasificado_t0"
estado_t1 = "estado_laboral_clasificado_t1"
ESTADOS_VALIDOS = ["Empleo Formal", "Empleo Informal", "Desocupación", "Inactividad"]

# Normalizar estados
panel[estado_t] = panel[estado_t].astype("string").str.strip()
panel[estado_t1] = panel[estado_t1].astype("string").str.strip()

# Ventana: 2023T4, 2024T1-T4, 2025T1-T3
mask_ventana = (
    ((panel["ANO4"] == 2023) & (panel["TRIMESTRE"] == 4)) |
    ((panel["ANO4"] == 2024) & (panel["TRIMESTRE"].between(1, 4))) |
    ((panel["ANO4"] == 2025) & (panel["TRIMESTRE"].between(1, 3)))
)
panel = panel[mask_ventana].copy()
print(f"Panel filtrado por ventana: {panel.shape[0]} filas")

# Restringir a origen = "Empleo Informal"
panel = panel[panel[estado_t] == "Empleo Informal"].copy()
print(f"Panel restringido a origen informal: {panel.shape[0]} filas")

# Excluir destinos no válidos
panel = panel[panel[estado_t1].isin(ESTADOS_VALIDOS)].copy()
print(f"Panel con destinos válidos: {panel.shape[0]} filas")

Panel filtrado por ventana: 145661 filas
Panel restringido a origen informal: 27916 filas
Panel con destinos válidos: 27891 filas


In [6]:
# 6. Merge con IVLR (clave: id_persona, ANO4, TRIMESTRE) - left join
CLAVES = ["id_persona", "ANO4", "TRIMESTRE"]
ivlr_cols = CLAVES + [
    "ivlr_basico",
    "ivlr_ampliado",
    "ivlr_ampliado_pesos_iguales",
    "cluster_final",
]

# Normalizar tipos de claves IVLR antes del merge
panel["id_persona"] = panel["id_persona"].astype("string").str.strip()
ivlr["id_persona"] = ivlr["id_persona"].astype("string").str.strip()

for columna in ["ANO4", "TRIMESTRE"]:
    ivlr[columna] = pd.to_numeric(ivlr[columna], errors="raise").astype("Int64")

print("\nTipos de claves del panel:")
print(panel[CLAVES].dtypes)

print("\nTipos de claves de la base IVLR:")
print(ivlr[CLAVES].dtypes)

ivlr_subset = ivlr[ivlr_cols].copy()

# Verificar que no hay duplicados en la clave de IVLR
duplicados_ivlr = int(ivlr_subset.duplicated(CLAVES).sum())
assert duplicados_ivlr == 0, f"La base IVLR tiene {duplicados_ivlr} claves duplicadas."

panel_analitico = panel.merge(
    ivlr_subset,
    on=CLAVES,
    how="left",
    validate="many_to_one",
    indicator=True,
)

print(f"Panel después del merge: {panel_analitico.shape[0]} filas")
print("_merge counts:\n", panel_analitico["_merge"].value_counts())

# Verificar que todos los casos se emparejaron (left join con todos los casos del panel)
assert len(panel_analitico) == 27891, f"Se esperaban 27891 filas, se obtuvieron {len(panel_analitico)}"
assert panel_analitico["_merge"].eq("both").all(), "No todos los casos se emparejaron (se esperaba left join exitoso)."


Tipos de claves del panel:
id_persona    string
ANO4           Int64
TRIMESTRE      Int64
dtype: object

Tipos de claves de la base IVLR:
id_persona    string
ANO4           Int64
TRIMESTRE      Int64
dtype: object
Panel después del merge: 27891 filas
_merge counts:
 _merge
both          27891
left_only         0
right_only        0
Name: count, dtype: int64


In [7]:
# 7. Controles obligatorios post-merge
expected_len = 27891
print(f"Longitud actual: {len(panel_analitico)}")
assert len(panel_analitico) == expected_len, f"Se esperaban {expected_len} filas, se obtuvieron {len(panel_analitico)}"
assert panel_analitico["_merge"].eq("both").all()
assert panel_analitico["ivlr_basico"].notna().sum() == 27184
assert panel_analitico["ivlr_ampliado"].notna().sum() == 22764
assert panel_analitico["ivlr_ampliado_pesos_iguales"].notna().sum() == 22764

Longitud actual: 27891


In [8]:
# 8. Ponderador estricto del trimestre de origen
assert "PONDERA" in panel_analitico.columns, (
    "No se encontró PONDERA del trimestre de origen."
)

panel_analitico["PONDERA_t0"] = pd.to_numeric(
    panel_analitico["PONDERA"],
    errors="coerce",
)

assert panel_analitico["PONDERA_t0"].notna().all()
assert panel_analitico["PONDERA_t0"].gt(0).all()

In [9]:
# 9. Función auxiliar: media ponderada
def media_ponderada(valores, pesos):
    datos = pd.DataFrame({
        "valor": pd.to_numeric(valores, errors="coerce"),
        "peso": pd.to_numeric(pesos, errors="coerce"),
    }).dropna()
    datos = datos[datos["peso"] > 0]
    if datos.empty:
        return np.nan
    return float(np.average(datos["valor"], weights=datos["peso"]))

In [10]:
# 10. Análisis 1: Distribución general de destinos
dist_destinos = (
    panel_analitico
    .groupby(estado_t1, dropna=False)
    .agg(
        n=("id_persona", "size"),
        poblacion_ponderada=("PONDERA_t0", "sum")
    )
    .reset_index()
)
dist_destinos["pct_no_ponderado"] = 100 * dist_destinos["n"] / dist_destinos["n"].sum()
dist_destinos["pct_ponderado"] = 100 * dist_destinos["poblacion_ponderada"] / dist_destinos["poblacion_ponderada"].sum()
dist_destinos = dist_destinos.sort_values(estado_t1)
print("Distribución de destinos:")
print(dist_destinos.to_string(index=False))

# Guardar
dist_destinos.to_csv(OUT_DIR / "distribucion_destinos_desde_informalidad.csv", index=False, encoding="utf-8-sig")

Distribución de destinos:
estado_laboral_clasificado_t1     n  poblacion_ponderada  pct_no_ponderado  pct_ponderado
                 Desocupación  1274             950503.0          4.567782       5.888219
                Empleo Formal  2917            1822954.0         10.458571      11.292918
              Empleo Informal 20193           11315653.0         72.399699      70.098720
                  Inactividad  3507            2053343.0         12.573949      12.720142


In [11]:
# 11. Análisis 2: IVLR inicial según destino
def resumen_ivlr_por_destino(df, ivlr_col):
    # Filtrar solo casos con IVLR observado
    temp = df[df[ivlr_col].notna()].copy()
    if temp.empty:
        return pd.DataFrame()
    res = temp.groupby(estado_t1).apply(
        lambda g: pd.Series({
            "n_total_destino": len(df[df[estado_t1] == g.name]),
            "n_con_ivlr": len(g),
            "pct_con_ivlr": 100 * len(g) / len(df[df[estado_t1] == g.name]),
            "media": g[ivlr_col].mean(),
            "p25": g[ivlr_col].quantile(0.25),
            "mediana": g[ivlr_col].median(),
            "p75": g[ivlr_col].quantile(0.75),
            "media_ponderada": media_ponderada(g[ivlr_col], g["PONDERA_t0"]),
        })
    ).reset_index().rename(columns={"index": estado_t1})
    res["ivlr_variable"] = ivlr_col
    return res

resumen_ivlr = pd.concat([
    resumen_ivlr_por_destino(panel_analitico, "ivlr_basico"),
    resumen_ivlr_por_destino(panel_analitico, "ivlr_ampliado"),
    resumen_ivlr_por_destino(panel_analitico, "ivlr_ampliado_pesos_iguales")
], ignore_index=True)

print("\nResumen IVLR inicial por destino:")
print(resumen_ivlr.to_string(index=False))
resumen_ivlr.to_csv(OUT_DIR / "resumen_ivlr_inicial_por_destino.csv", index=False, encoding="utf-8-sig")


Resumen IVLR inicial por destino:
estado_laboral_clasificado_t1  n_total_destino  n_con_ivlr  pct_con_ivlr    media      p25  mediana      p75  media_ponderada               ivlr_variable
                 Desocupación           1274.0      1231.0     96.624804 0.379041 0.000000 0.300000 0.700000         0.408042                 ivlr_basico
                Empleo Formal           2917.0      2867.0     98.285910 0.176317 0.000000 0.000000 0.300000         0.171584                 ivlr_basico
              Empleo Informal          20193.0     19790.0     98.004259 0.263259 0.000000 0.300000 0.300000         0.270420                 ivlr_basico
                  Inactividad           3507.0      3296.0     93.983462 0.288167 0.000000 0.300000 0.700000         0.309360                 ivlr_basico
                 Desocupación           1274.0      1020.0     80.062794 0.471946 0.272964 0.439701 0.693773         0.503379               ivlr_ampliado
                Empleo Formal           2

In [12]:
# 12. Análisis 3: Cuartiles del IVLR ampliado (dentro de cada trimestre de origen)
df_temp = panel_analitico[panel_analitico["ivlr_ampliado"].notna()].copy()
df_temp["cuartil_ivlr_ampliado"] = (
    df_temp.groupby(["ANO4", "TRIMESTRE"])["ivlr_ampliado"]
    .transform(lambda x: pd.qcut(x.rank(method="first"), q=4, labels=["Q1 menor vulnerabilidad", "Q2", "Q3", "Q4 mayor vulnerabilidad"]))
)
assert df_temp["cuartil_ivlr_ampliado"].notna().all(), "Hay NA en cuartil para casos con IVLR observado."

# Merge de vuelta al dataset completo (con IVLR NA)
panel_analitico = panel_analitico.merge(
    df_temp[["id_persona", "ANO4", "TRIMESTRE", "cuartil_ivlr_ampliado"]],
    on=["id_persona", "ANO4", "TRIMESTRE"],
    how="left"
)

# Crear panel_cuartiles para análisis con cuartil no NA
panel_cuartiles = panel_analitico.loc[
    panel_analitico["cuartil_ivlr_ampliado"].notna()
].copy()

# Control adicional post-cuartiles
assert len(panel_cuartiles) == 22764

conteos_cuartil_periodo = (
    panel_cuartiles
    .groupby(["ANO4", "TRIMESTRE", "cuartil_ivlr_ampliado"], observed=True)
    .size()
    .reset_index(name="n")
)

assert conteos_cuartil_periodo["n"].gt(0).all()

# Distribución de destinos por cuartil (solo para casos con cuartil no NA)
destinos_cuartil = (
    panel_cuartiles
    .groupby(["cuartil_ivlr_ampliado", estado_t1], dropna=False)
    .agg(
        n=("id_persona", "size"),
        poblacion_ponderada=("PONDERA_t0", "sum")
    )
    .reset_index()
)
destinos_cuartil["pct_no_ponderado"] = destinos_cuartil.groupby("cuartil_ivlr_ampliado")["n"].transform(lambda x: 100 * x / x.sum())
destinos_cuartil["pct_ponderado"] = destinos_cuartil.groupby("cuartil_ivlr_ampliado")["poblacion_ponderada"].transform(lambda x: 100 * x / x.sum())
destinos_cuartil = destinos_cuartil.sort_values(["cuartil_ivlr_ampliado", estado_t1])

print("\nDestinos por cuartil de IVLR ampliado:")
print(destinos_cuartil.to_string(index=False))
destinos_cuartil.to_csv(OUT_DIR / "destinos_por_cuartil_ivlr_ampliado.csv", index=False, encoding="utf-8-sig")


Destinos por cuartil de IVLR ampliado:
  cuartil_ivlr_ampliado estado_laboral_clasificado_t1    n  poblacion_ponderada  pct_no_ponderado  pct_ponderado
Q1 menor vulnerabilidad                  Desocupación  172             116486.0          3.020724       3.455816
Q1 menor vulnerabilidad                 Empleo Formal 1017             644209.0         17.860906      19.111894
Q1 menor vulnerabilidad               Empleo Informal 4104            2396678.0         72.075869      71.102787
Q1 menor vulnerabilidad                   Inactividad  401             213350.0          7.042501       6.329503
                     Q2                  Desocupación  198             111690.0          3.479178       3.899068
                     Q2                 Empleo Formal  532             284733.0          9.348093       9.939952
                     Q2               Empleo Informal 4465            2228754.0         78.457213      77.805197
                     Q2                   Inactividad  4

In [14]:
# 13. Análisis 4: Resultados binarios por cuartil
# Crear variables binarias (solo para descripción)
panel_analitico["formalizacion_t1"] = (panel_analitico[estado_t1] == "Empleo Formal").astype("Int64")
panel_analitico["persistencia_informal_t1"] = (panel_analitico[estado_t1] == "Empleo Informal").astype("Int64")
panel_analitico["salida_no_empleo_t1"] = panel_analitico[estado_t1].isin(["Desocupación", "Inactividad"]).astype("Int64")

# panel_cuartiles es una copia independiente creada en el análisis anterior;
# replicar en ella las variables binarias para que la función las encuentre.
panel_cuartiles["formalizacion_t1"] = (panel_cuartiles[estado_t1] == "Empleo Formal").astype("Int64")
panel_cuartiles["persistencia_informal_t1"] = (panel_cuartiles[estado_t1] == "Empleo Informal").astype("Int64")
panel_cuartiles["salida_no_empleo_t1"] = panel_cuartiles[estado_t1].isin(["Desocupación", "Inactividad"]).astype("Int64")

def resumir_resultados_binarios(grupo):
    """Calcula métricas binarias para un grupo (cuartil)."""
    n = len(grupo)
    formal = grupo["formalizacion_t1"].mean()
    persist = grupo["persistencia_informal_t1"].mean()
    salida = grupo["salida_no_empleo_t1"].mean()
    formal_pond = media_ponderada(grupo["formalizacion_t1"], grupo["PONDERA_t0"])
    persist_pond = media_ponderada(grupo["persistencia_informal_t1"], grupo["PONDERA_t0"])
    salida_pond = media_ponderada(grupo["salida_no_empleo_t1"], grupo["PONDERA_t0"])
    return pd.Series({
        "n": n,
        "formalizacion": formal,
        "persistencia_informal": persist,
        "salida_no_empleo": salida,
        "formalizacion_pond": formal_pond,
        "persistencia_pond": persist_pond,
        "salida_pond": salida_pond,
    })

resultados_binarios = (
    panel_cuartiles
    .groupby("cuartil_ivlr_ampliado", observed=True)
    .apply(resumir_resultados_binarios, include_groups=False)
    .reset_index()
)

print("\nResultados binarios por cuartil de IVLR ampliado:")
print(resultados_binarios.to_string(index=False))
resultados_binarios.to_csv(OUT_DIR / "resultados_binarios_por_cuartil_ivlr.csv", index=False, encoding="utf-8-sig")


Resultados binarios por cuartil de IVLR ampliado:
  cuartil_ivlr_ampliado      n  formalizacion  persistencia_informal  salida_no_empleo  formalizacion_pond  persistencia_pond  salida_pond
Q1 menor vulnerabilidad 5694.0       0.178609               0.720759          0.100632            0.191119           0.711028     0.097853
                     Q2 5691.0       0.093481               0.784572          0.121947            0.099400           0.778052     0.122549
                     Q3 5688.0       0.072609               0.733298          0.194093            0.079222           0.712384     0.208394
Q4 mayor vulnerabilidad 5691.0       0.056053               0.698296          0.245651            0.051118           0.663105     0.285777


In [15]:
# 14. Análisis 5: Comparación de especificaciones (pesos iguales y ivlr_basico)
# 14a. Cuartiles de ivlr_ampliado_pesos_iguales
df_temp2 = panel_analitico[panel_analitico["ivlr_ampliado_pesos_iguales"].notna()].copy()
df_temp2["cuartil_ivlr_pesos_iguales"] = (
    df_temp2.groupby(["ANO4", "TRIMESTRE"])["ivlr_ampliado_pesos_iguales"]
    .transform(lambda x: pd.qcut(x.rank(method="first"), q=4, labels=["Q1", "Q2", "Q3", "Q4"]))
)
panel_analitico = panel_analitico.merge(
    df_temp2[["id_persona", "ANO4", "TRIMESTRE", "cuartil_ivlr_pesos_iguales"]],
    on=["id_persona", "ANO4", "TRIMESTRE"],
    how="left"
)

panel_cuartiles_iguales = panel_analitico.loc[
    panel_analitico["cuartil_ivlr_pesos_iguales"].notna()
].copy()

destinos_pesos_iguales = (
    panel_cuartiles_iguales
    .groupby(["cuartil_ivlr_pesos_iguales", estado_t1], dropna=False)
    .agg(
        n=("id_persona", "size"),
        poblacion_ponderada=("PONDERA_t0", "sum")
    )
    .reset_index()
)
destinos_pesos_iguales["pct_no_ponderado"] = destinos_pesos_iguales.groupby("cuartil_ivlr_pesos_iguales")["n"].transform(lambda x: 100 * x / x.sum())
destinos_pesos_iguales["pct_ponderado"] = destinos_pesos_iguales.groupby("cuartil_ivlr_pesos_iguales")["poblacion_ponderada"].transform(lambda x: 100 * x / x.sum())
destinos_pesos_iguales = destinos_pesos_iguales.sort_values(["cuartil_ivlr_pesos_iguales", estado_t1])

print("\nDestinos por cuartil de IVLR con pesos iguales:")
print(destinos_pesos_iguales.to_string(index=False))
destinos_pesos_iguales.to_csv(OUT_DIR / "sensibilidad_destinos_pesos_iguales.csv", index=False, encoding="utf-8-sig")

# 14b. ivlr_basico (no es un conteo, sino una combinación ponderada de intensidad y educación)
# Se agrupa por los valores discretos observados (0, 0.3, 0.7, 1.0)
ivlr_basico_groups = (
    panel_analitico[panel_analitico["ivlr_basico"].notna()]
    .groupby(["ivlr_basico", estado_t1], dropna=False)
    .agg(
        n=("id_persona", "size"),
        poblacion_ponderada=("PONDERA_t0", "sum")
    )
    .reset_index()
)
ivlr_basico_groups["pct_no_ponderado"] = ivlr_basico_groups.groupby("ivlr_basico")["n"].transform(lambda x: 100 * x / x.sum())
ivlr_basico_groups["pct_ponderado"] = ivlr_basico_groups.groupby("ivlr_basico")["poblacion_ponderada"].transform(lambda x: 100 * x / x.sum())
ivlr_basico_groups = ivlr_basico_groups.sort_values(["ivlr_basico", estado_t1])

print("\nDestinos por valor de ivlr_basico:")
print(ivlr_basico_groups.to_string(index=False))
ivlr_basico_groups.to_csv(OUT_DIR / "destinos_por_valor_ivlr_basico.csv", index=False, encoding="utf-8-sig")


Destinos por cuartil de IVLR con pesos iguales:
cuartil_ivlr_pesos_iguales estado_laboral_clasificado_t1    n  poblacion_ponderada  pct_no_ponderado  pct_ponderado
                        Q1                  Desocupación  170             115308.0          2.985599       3.427488
                        Q1                 Empleo Formal 1016             639665.0         17.843344      19.013808
                        Q1               Empleo Informal 4108            2395624.0         72.146119      71.209047
                        Q1                   Inactividad  400             213616.0          7.024939       6.349657
                        Q2                  Desocupación  238             147114.0          4.182042       5.197886
                        Q2                 Empleo Formal  509             282310.0          8.943947       9.974681
                        Q2               Empleo Informal 4049            1940299.0         71.147426      68.555358
                       

In [16]:
# 15. Análisis 6: Control temporal
def resumir_periodo(grupo):
    """Resume métricas por período (ANO4, TRIMESTRE)."""
    n = len(grupo)
    n_ivlr = grupo["ivlr_ampliado"].notna().sum()
    cobertura = 100 * n_ivlr / n if n > 0 else np.nan
    media_ivlr = grupo["ivlr_ampliado"].mean()
    mediana_ivlr = grupo["ivlr_ampliado"].median()
    formal = grupo["formalizacion_t1"].mean()
    persist = grupo["persistencia_informal_t1"].mean()
    salida = grupo["salida_no_empleo_t1"].mean()
    formal_pond = media_ponderada(grupo["formalizacion_t1"], grupo["PONDERA_t0"])
    persist_pond = media_ponderada(grupo["persistencia_informal_t1"], grupo["PONDERA_t0"])
    salida_pond = media_ponderada(grupo["salida_no_empleo_t1"], grupo["PONDERA_t0"])
    return pd.Series({
        "n_enlaces": n,
        "n_ivlr_ampliado": n_ivlr,
        "cobertura_ivlr_ampliado": cobertura,
        "media_ivlr_ampliado": media_ivlr,
        "mediana_ivlr_ampliado": mediana_ivlr,
        "formalizacion_no_pond": formal,
        "persistencia_no_pond": persist,
        "salida_no_empleo_no_pond": salida,
        "formalizacion_pond": formal_pond,
        "persistencia_pond": persist_pond,
        "salida_pond": salida_pond,
    })

control_temporal = (
    panel_analitico
    .groupby(["ANO4", "TRIMESTRE"])
    .apply(resumir_periodo, include_groups=False)
    .reset_index()
)

print("\nControl temporal:")
print(control_temporal.to_string(index=False))
control_temporal.to_csv(OUT_DIR / "transiciones_ivlr_por_periodo.csv", index=False, encoding="utf-8-sig")


Control temporal:
 ANO4  TRIMESTRE  n_enlaces  n_ivlr_ampliado  cobertura_ivlr_ampliado  media_ivlr_ampliado  mediana_ivlr_ampliado  formalizacion_no_pond  persistencia_no_pond  salida_no_empleo_no_pond  formalizacion_pond  persistencia_pond  salida_pond
 2023          4     3323.0           2863.0                86.157087             0.377928               0.340444               0.104424              0.702377                  0.193199            0.103217           0.683438     0.213346
 2024          1     3578.0           2911.0                81.358301             0.378873               0.340463               0.106764              0.709335                  0.183902            0.110652           0.694689     0.194659
 2024          2     3460.0           2896.0                83.699422             0.376332               0.341153               0.100289              0.728035                  0.171676            0.102243           0.720016     0.177741
 2024          3     3686.0      

In [17]:
# 16. Análisis 7: Control por cluster (contexto descriptivo)
# 16a. Control de cobertura y IVLR por cluster
control_cluster = (
    panel_analitico
    .groupby("cluster_final", dropna=False)
    .agg(
        n_enlaces=("id_persona", "size"),
        n_ivlr_ampliado=("ivlr_ampliado", lambda x: x.notna().sum()),
        cobertura_ivlr_ampliado=("ivlr_ampliado", lambda x: 100 * x.notna().mean()),
        media_ivlr_ampliado=("ivlr_ampliado", "mean"),
        mediana_ivlr_ampliado=("ivlr_ampliado", "median"),
    )
    .reset_index()
)

# 16b. Destinos por cluster
destinos_cluster_raw = (
    panel_analitico
    .groupby(["cluster_final", estado_t1], dropna=False)
    .agg(
        n=("id_persona", "size"),
        poblacion_ponderada=("PONDERA_t0", "sum")
    )
    .reset_index()
)
destinos_cluster_raw["pct_no_ponderado"] = (
    destinos_cluster_raw
    .groupby("cluster_final", dropna=False)["n"]
    .transform(lambda x: 100 * x / x.sum())
)
destinos_cluster_raw["pct_ponderado"] = (
    destinos_cluster_raw
    .groupby("cluster_final", dropna=False)["poblacion_ponderada"]
    .transform(lambda x: 100 * x / x.sum())
)
destinos_cluster_raw = destinos_cluster_raw.sort_values(["cluster_final", estado_t1])

# 16c. Unir ambos para exportar una tabla completa
destinos_cluster = destinos_cluster_raw.merge(
    control_cluster,
    on="cluster_final",
    how="left"
)

print("\nDestinos por cluster:")
print(destinos_cluster.to_string(index=False))
destinos_cluster.to_csv(OUT_DIR / "transiciones_ivlr_por_cluster.csv", index=False, encoding="utf-8-sig")


Destinos por cluster:
 cluster_final estado_laboral_clasificado_t1    n  poblacion_ponderada  pct_no_ponderado  pct_ponderado  n_enlaces  n_ivlr_ampliado  cobertura_ivlr_ampliado  media_ivlr_ampliado  mediana_ivlr_ampliado
             1                  Desocupación  409             298763.0          4.778596       6.205432       8559             6761                78.992873             0.424055                0.39308
             1                 Empleo Formal  742             417207.0          8.669237       8.665563       8559             6761                78.992873             0.424055                0.39308
             1               Empleo Informal 6108            3347135.0         71.363477      69.521387       8559             6761                78.992873             0.424055                0.39308
             1                   Inactividad 1300             751435.0          15.18869      15.607618       8559             6761                78.992873             0.42

In [18]:
# 17. Exportar controles del enlace y archivos adicionales
control_enlace = pd.DataFrame({
    "indicador": [
        "enlaces_totales_panel",
        "enlaces_emparejados",
        "ivlr_basico_observado",
        "ivlr_ampliado_observado",
        "ivlr_pesos_iguales_observado"
    ],
    "valor": [
        len(panel_analitico),
        len(panel_analitico[panel_analitico["_merge"] == "both"]),
        panel_analitico["ivlr_basico"].notna().sum(),
        panel_analitico["ivlr_ampliado"].notna().sum(),
        panel_analitico["ivlr_ampliado_pesos_iguales"].notna().sum()
    ]
})
control_enlace.to_csv(OUT_DIR / "control_enlace_ivlr_transiciones.csv", index=False, encoding="utf-8-sig")

In [24]:
# 18. Conclusiones y texto final
texto = (
    "EVIDENCIA EMPÍRICA:\n"
    "- Se analizaron 27.891 transiciones desde empleo informal (ventana 2023T4-2025T3).\n"
    "- La cobertura del IVLR ampliado es del {:.1f}% ({:,} casos).\n"
    "- La distribución de destinos se encuentra en las tablas exportadas.\n"
    "- Las comparaciones por cuartiles del IVLR inicial y su relación con los destinos\n"
    "  están disponibles en las tablas adjuntas.\n\n"
    "INTERPRETACIÓN METODOLÓGICA:\n"
    "- Las distribuciones de destino se compararon según el IVLR inicial y sus cuartiles trimestrales.\n"
    "- La dirección y magnitud de las asociaciones deben interpretarse a partir de las tablas exportadas.\n"
    "- Las diferencias observadas no representan efectos causales.\n"
    "- El IVLR fue construido exclusivamente con información de t, sin usar transiciones.\n"
    "- El análisis permite evaluar si el IVLR inicial presenta una asociación prospectiva con los destinos laborales posteriores.\n\n"
    "HIPÓTESIS DE TRABAJO (a ser evaluada en el ABM):\n"
    "- Un IVLR inicial más alto podría asociarse con mayor persistencia informal y menor formalización,\n"
    "  lo que sería consistente con la teoría de segmentación laboral.\n"
    "- El ABM utilizará posteriormente estas regularidades empíricas como insumo de calibración y contraste,\n"
    "  sin interpretarlas automáticamente como relaciones causales.\n\n"
    "Un IVLR inicial más alto se asocia descriptivamente con menor formalización y mayor salida posterior hacia desocupación o inactividad. "
    "La persistencia informal no presenta un gradiente estrictamente monotónico."
    "NOTA: Este análisis no modifica el IVLR ni sus pesos, y no utiliza información de t+1 para construirlo."
).format(
    100 * panel_analitico["ivlr_ampliado"].notna().sum() / len(panel_analitico),
    panel_analitico["ivlr_ampliado"].notna().sum()
)

with open(OUT_DIR / "conclusiones_transiciones_ivlr.txt", "w", encoding="utf-8") as f:
    f.write(texto)

print(texto)

EVIDENCIA EMPÍRICA:
- Se analizaron 27.891 transiciones desde empleo informal (ventana 2023T4-2025T3).
- La cobertura del IVLR ampliado es del 81.6% (22,764 casos).
- La distribución de destinos se encuentra en las tablas exportadas.
- Las comparaciones por cuartiles del IVLR inicial y su relación con los destinos
  están disponibles en las tablas adjuntas.

INTERPRETACIÓN METODOLÓGICA:
- Las distribuciones de destino se compararon según el IVLR inicial y sus cuartiles trimestrales.
- La dirección y magnitud de las asociaciones deben interpretarse a partir de las tablas exportadas.
- Las diferencias observadas no representan efectos causales.
- El IVLR fue construido exclusivamente con información de t, sin usar transiciones.
- El análisis permite evaluar si el IVLR inicial presenta una asociación prospectiva con los destinos laborales posteriores.

HIPÓTESIS DE TRABAJO (a ser evaluada en el ABM):
- Un IVLR inicial más alto podría asociarse con mayor persistencia informal y menor forma

In [20]:
# 19. Versiones del entorno
import sys, pandas, numpy, pyarrow, platform
versiones = pd.DataFrame({
    "componente": ["Python", "pandas", "numpy", "pyarrow", "sistema", "entorno_conda"],
    "version": [
        sys.version.split()[0],
        pd.__version__,
        np.__version__,
        pa.__version__,
        platform.system(),
        os.environ.get("CONDA_DEFAULT_ENV", "desconocido")
    ]
})
versiones.to_csv(OUT_DIR / "versiones_entorno_transiciones_ivlr.csv", index=False, encoding="utf-8-sig")
print("\nVersiones del entorno guardadas.")


Versiones del entorno guardadas.


In [21]:
# 20. Controles finales
assert len(panel_analitico) == 27891
assert panel_analitico[estado_t1].isin(ESTADOS_VALIDOS).all()
suma_binarios = panel_analitico["formalizacion_t1"] + panel_analitico["persistencia_informal_t1"] + panel_analitico["salida_no_empleo_t1"]
assert (suma_binarios == 1).all()
assert dist_destinos["n"].sum() == 27891

# Controles de suma por cuartil
control_suma_cuartiles = (
    destinos_cuartil
    .groupby("cuartil_ivlr_ampliado", observed=True)
    .agg(
        suma_pct_no_ponderado=("pct_no_ponderado", "sum"),
        suma_pct_ponderado=("pct_ponderado", "sum"),
    )
    .reset_index()
)

assert np.allclose(control_suma_cuartiles["suma_pct_no_ponderado"], 100, atol=1e-8)
assert np.allclose(control_suma_cuartiles["suma_pct_ponderado"], 100, atol=1e-8)
assert destinos_cuartil["n"].sum() == 22764
assert resultados_binarios["n"].sum() == 22764
assert destinos_pesos_iguales["n"].sum() == 22764

print("\nControles finales superados.")


Controles finales superados.
